# megane - Jupyter Widget Demo

A fast, beautiful molecular viewer for Jupyter notebooks.
Built with [anywidget](https://anywidget.dev/) — works in JupyterLab, Jupyter Notebook, VS Code, and Google Colab.

In [ ]:
import megane

print(f"megane v{megane.__version__}")

## Basic Usage

Load a PDB file and display the 3D viewer. The viewer supports mouse interaction:
- **Left-drag**: Rotate
- **Scroll**: Zoom
- **Right-click atom**: Select / measure distances, angles, dihedrals

In [ ]:
pipe = megane.Pipeline()
s = pipe.add_node(megane.LoadStructure("../tests/fixtures/caffeine_water.pdb"))
bonds = pipe.add_node(megane.AddBonds(source="distance"))
v = pipe.add_node(megane.Viewport())

pipe.add_edge(s.out.particle, bonds.inp.particle)
pipe.add_edge(s.out.particle, v.inp.particle)
pipe.add_edge(bonds.out.bond, v.inp.bond)

viewer = megane.MolecularViewer()
viewer.set_pipeline(pipe)
viewer

## Trajectory Playback

Load a structure together with an XTC trajectory file. Use `frame_index` to navigate frames programmatically.

In [ ]:
traj_pipe = megane.Pipeline()
s = traj_pipe.add_node(megane.LoadStructure("../tests/fixtures/caffeine_water.pdb"))
t = traj_pipe.add_node(megane.LoadTrajectory(xtc="../tests/fixtures/caffeine_water_vibration.xtc"))
bonds = traj_pipe.add_node(megane.AddBonds(source="structure"))
v = traj_pipe.add_node(megane.Viewport())

traj_pipe.add_edge(s.out.particle, t.inp.particle)
traj_pipe.add_edge(s.out.particle, bonds.inp.particle)
traj_pipe.add_edge(s.out.particle, v.inp.particle)
traj_pipe.add_edge(t.out.traj, v.inp.traj)
traj_pipe.add_edge(bonds.out.bond, v.inp.bond)

traj_viewer = megane.MolecularViewer()
traj_viewer.set_pipeline(traj_pipe)
print(f"Loaded trajectory: {traj_viewer.total_frames} frames")
traj_viewer

In [ ]:
# Jump to a specific frame
traj_viewer.frame_index = 50

In [ ]:
# Animate through frames
import time

for i in range(0, traj_viewer.total_frames, 2):
    traj_viewer.frame_index = i
    time.sleep(0.05)

## Multiple Viewers

Display multiple viewers side-by-side using `ipywidgets.HBox`. Each viewer is independent.

In [ ]:
import ipywidgets as widgets

pipe1 = megane.Pipeline()
s1 = pipe1.add_node(megane.LoadStructure("../tests/fixtures/caffeine_water.pdb"))
b1 = pipe1.add_node(megane.AddBonds(source="distance"))
vp1 = pipe1.add_node(megane.Viewport())
pipe1.add_edge(s1.out.particle, b1.inp.particle)
pipe1.add_edge(s1.out.particle, vp1.inp.particle)
pipe1.add_edge(b1.out.bond, vp1.inp.bond)

pipe2 = megane.Pipeline()
s2 = pipe2.add_node(megane.LoadStructure("../tests/fixtures/caffeine_water.pdb"))
t2 = pipe2.add_node(megane.LoadTrajectory(xtc="../tests/fixtures/caffeine_water_vibration.xtc"))
b2 = pipe2.add_node(megane.AddBonds(source="structure"))
vp2 = pipe2.add_node(megane.Viewport())
pipe2.add_edge(s2.out.particle, t2.inp.particle)
pipe2.add_edge(s2.out.particle, b2.inp.particle)
pipe2.add_edge(s2.out.particle, vp2.inp.particle)
pipe2.add_edge(t2.out.traj, vp2.inp.traj)
pipe2.add_edge(b2.out.bond, vp2.inp.bond)

v1 = megane.MolecularViewer()
v1.set_pipeline(pipe1)

v2 = megane.MolecularViewer()
v2.set_pipeline(pipe2)

widgets.HBox([v1, v2])